# Project 39 — LangGraph Ticket Escalation Router
## Auto-resolve simple tickets or escalate complex ones

**Stack:** LangGraph · Ollama · Jupyter

**Workflow:** Classify Ticket → Attempt Auto Resolve → Escalate → Send Response

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

class WorkflowState(TypedDict):
    input_data: str
    accumulated: str
    final_output: str

## Step 2 — Define Workflow Nodes

In [ ]:
def classify_ticket(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 1 of a LangGraph Ticket Escalation Router pipeline. "
         "Your role: classify ticket. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[classify_ticket]: " + result
    print(f"  Step 1/4: classify_ticket")
    return {"accumulated": accumulated, "final_output": result}

def attempt_auto_resolve(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 2 of a LangGraph Ticket Escalation Router pipeline. "
         "Your role: attempt auto resolve. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[attempt_auto_resolve]: " + result
    print(f"  Step 2/4: attempt_auto_resolve")
    return {"accumulated": accumulated, "final_output": result}

def escalate(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 3 of a LangGraph Ticket Escalation Router pipeline. "
         "Your role: escalate. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[escalate]: " + result
    print(f"  Step 3/4: escalate")
    return {"accumulated": accumulated, "final_output": result}

def send_response(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are step 4 of a LangGraph Ticket Escalation Router pipeline. "
         "Your role: send response. "
         "Process the input and produce structured output for the next step."),
        ("human", "Input: {input}\nPrevious steps: {context}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"input": state["input_data"], "context": state.get("accumulated", "")})
    accumulated = state.get("accumulated", "") + f"\n\n[send_response]: " + result
    print(f"  Step 4/4: send_response")
    return {"accumulated": accumulated, "final_output": result}

## Step 3 — Build and Compile Graph

In [ ]:
graph = StateGraph(WorkflowState)
graph.add_node("classify_ticket", classify_ticket)
graph.add_node("attempt_auto_resolve", attempt_auto_resolve)
graph.add_node("escalate", escalate)
graph.add_node("send_response", send_response)

graph.set_entry_point("classify_ticket")
graph.add_edge("classify_ticket", "attempt_auto_resolve")
graph.add_edge("attempt_auto_resolve", "escalate")
graph.add_edge("escalate", "send_response")
graph.add_edge("send_response", END)

app = graph.compile()
print("LangGraph Ticket Escalation Router — workflow compiled!")

## Step 4 — Run the Workflow

In [ ]:
sample_input = "Analyze and process this request through the LangGraph Ticket Escalation Router pipeline."
result = app.invoke({
    "input_data": sample_input, "accumulated": "", "final_output": "",
})

print("=== WORKFLOW RESULT ===")
print(result["final_output"])
print("\n=== FULL TRACE ===")
print(result["accumulated"][:1000])

## What You Learned
- **Multi-node LangGraph workflow** for auto-resolve simple tickets or escalate complex ones
- **Progressive context accumulation** across steps
- **Structured pipeline execution** with trace logging